# Chapter 9: Reinforcement Learning. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/09-rl.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/09-rl.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/09-rl.ipynb)

In [ ]:
# Packages that the exercises of this notebook use; the ones that are missing are installed
pkgs <- c("laGP")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org")

## Exercise 9.7: Multi-Armed Bandit: Thompson Sampling

A website tests 3 button colors: Red (A), Green (B), Blue (C). Each has an unknown click-through rate $\theta_i$. After some initial testing, the posterior distributions are:

- Red: Beta(15, 85)
- Green: Beta(22, 78)
- Blue: Beta(18, 82)

a) Calculate the posterior mean click-through rate for each button.

b) Using Thompson Sampling, simulate which button would be selected in the next trial.

c) Repeat part (b) 1000 times and report the selection frequencies.

d) After 100 more trials with 25 clicks on Green, update the Green posterior and recalculate.

## Exercise 9.8: Multi-Armed Bandit A/B Test

Consider a complex A/B experiment, with 6 alternatives, when you have 5 variations to your page, plus the original.

a) Use Bonferroni correction for multiple comparisons. Calculate the significance level for each of the 5 tests and find the number of samples per arm needed to achieve a power of 0.95, in a two-sided test, against an improvement from the original's 4% conversion rate to 5%. Assume that the overall significance level is .05.

b) Implement TS for the same experiment, with 100 visits per day. Assume an original arm with a 4% conversion rate, and an optimal arm with a 5% conversion rate. The other 4 arms include one suboptimal arm that beats the original with a conversion rate of 4.5%, and three inferior arms with rates of 3%, 2%, and 3.5%. End the experiment when the potential value remaining, the stopping rule of this chapter, falls below 1% of the champion's conversion rate, or after 919 days, the length of the classical test, whichever comes first. Plot the savings from a six-armed experiment, relative to a Bonferroni adjusted power calculation for a classical experiment. First plot should show the number of days required to end the experiment, with the vertical line showing the time required by the classical power calculation. The second plot should show the number of conversions that were saved by the bandit. What is the overall cost savings due to ending the experiment more quickly, and due to the experiment being less wasteful while it is running?

c) Run your simulator 500 times and show the history of the serving weights for all the arms in the first of our 500 simulation runs. Comment on the results.

d) Plot the daily cost of running the multi-armed bandit relative to an "oracle" strategy of always playing arm 2, the optimal arm.

## Exercise 9.9: Contextual Bandits: Weekday versus Weekend

The chapter's Contextual Bandits section describes, without ever simulating it, an experiment in which "Arm A performs slightly better during the weekdays, while Arm B excels during the weekends." Make this concrete. Let $x_t=1$ denote a weekend, with $\mathrm{P}\left(x_t=1\right)=2/7$, and $x_t=0$ a weekday. The true click-through rates are: arm A is $0.06$ on weekdays and $0.03$ on weekends; arm B is $0.03$ on weekdays and $0.08$ on weekends. A fresh context $x_t$ is drawn each round.

a) Implement a context-blind Thompson sampling bandit that keeps a single $Beta(1,1)$ posterior per arm (as in the chapter's `bandit()` function) and never observes $x_t$.

b) Implement a context-aware Thompson sampling bandit that keeps a separate $Beta(1,1)$ posterior for each of the four (arm, context) cells. Explain why this is the Bayesian-conjugate analogue of the chapter's logistic bandit model $\mathrm{logit}\,\mathrm{P}\left(\text{click}_a \mid \theta, x\right) = \theta_{0a} + \beta^\top x$ only once $\beta$ is allowed to depend on the arm (an arm-by-context interaction, $\beta_a$), and why the shared-$\beta$ version as literally written in the chapter cannot represent a crossover in which different arms are optimal in different contexts.

c) Simulate $T=5000$ rounds for each policy and plot cumulative regret for both on the same axes, where the regret at round $t$ is $\max(\theta_A(x_t),\theta_B(x_t)) - \theta_{\text{chosen}}(x_t)$.

d) Report the final cumulative regret of each policy and their ratio. In terms of the blended (marginal) rates $\bar\theta_a = \mathrm{P}\left(x=0\right)\theta_{a,\text{weekday}} + \mathrm{P}\left(x=1\right)\theta_{a,\text{weekend}}$, explain which arm the context-blind bandit converges to playing almost exclusively, and why. By extending the simulation to $T=50{,}000$, argue whether the context-blind bandit's regret grows linearly or sub-linearly in $T$, and contrast with the context-aware bandit and the chapter's $O(\log T)$ Thompson sampling claim.

e) Using only the (arm, context, outcome) triples logged by the context-blind bandit, fit `glm(y ~ arm * ctx, family = binomial)`. Show that it recovers the true crossover even though the policy that generated the data never used context to choose an arm. Explain why this does not help the context-blind bandit's own regret, and what it implies about needing context in the decision rule itself rather than only in a post-hoc analysis.

## Exercise 9.10: Bayesian Optimization: Expected Improvement

Continue the chapter's Taxi Fleet Optimisation example (Example 9.7): a GP surrogate is fit to the initial design $x=(10,30,90)$ with observed profits $(3.1,3.6,6.6)$, using the chapter's squared-exponential kernel and hyperparameters, on $y=-\text{profit}$ so that $x^*=\arg\min_x f(x)$.

a) Fit the GP to these three points and extract the posterior mean $\mu(x)$ and standard deviation $\sigma(x)$ at two candidate fleet sizes, $x=50$ and $x=70$.

b) By hand, i.e., directly from the closed-form formula rather than a library call, compute the Expected Improvement at each candidate,
$$
z = \frac{f^*-\mu(x)}{\sigma(x)}, \qquad EI(x) = \big(f^*-\mu(x)\big)\Phi(z) + \sigma(x)\phi(z),
$$
where $f^*=\min(y_1,\ldots,y_n)$ is the best (most negative) value observed so far and $\phi,\Phi$ are the standard normal density and CDF. Which candidate does Bayesian optimization sample next, and why?

c) Since the actual taxi simulator is not available outside the Emukit demo, implement the full Bayesian optimization loop (fit GP, maximize $EI$ over a grid, evaluate, update) on the synthetic profit curve $\text{profit}(x)=9\exp\big(-(x-45)^2/1250\big)$, which peaks at essentially the same fleet size the chapter's own search converged to. Starting from the same three points, run 5 iterations. Does the search find the true optimum $x^*=45$?